# Digit Recognizer — PyTorch Starter (Clean & Reproducible)

This notebook sets up a **reproducible PyTorch environment** for the Digit Recognizer competition.  
Key goals:
- deterministic training with fixed seeds,
- clear device selection (CPU/GPU),
- minimal, readable imports only,
- comments that explain *why* each step exists.

> From here you can plug in: dataset loading (CSV → tensors), a small CNN, training loop, and submission generation.  
> Keep the code modular and keep **`submission.csv`** in `/kaggle/working` to submit directly.


In [ ]:
# ================================
# Core setup: imports & reproducibility
# ================================

# Standard libs
import os, random, math, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim

# TorchVision (transforms/utilities; the dataset here comes from CSVs)
import torchvision
import torchvision.transforms as transforms

from torch.utils.data import Dataset, DataLoader, random_split

# ----------------
# Reproducibility
# ----------------
SEED = 42

def set_seed(seed: int = SEED) -> None:
    """
    Make training as deterministic as possible across Python, NumPy, and PyTorch.
    Note: complete determinism can reduce performance on GPU.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # cuDNN determinism settings
    torch.backends.cudnn.deterministic = True   # use deterministic algorithms where available
    torch.backends.cudnn.benchmark = False      # avoid non-deterministic autotuner paths

set_seed(SEED)

# ----------------
# Device selection
# ----------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(
    f"Device: {DEVICE} | "
    f"Torch: {torch.__version__} | "
    f"TorchVision: {torchvision.__version__}"
)

# Tip: if you need faster training at the cost of non-determinism, set:
# torch.backends.cudnn.deterministic = False
# torch.backends.cudnn.benchmark = True



## Notes

- **Why seeding?** Ensures runs are comparable (very useful when you tweak augmentation, model depth, or hyperparameters).
- **Why `cudnn.deterministic=True`?** Enforces deterministic GPU kernels (trade-off: slightly slower).
- **Next steps in the notebook**
  1. Load `train.csv` / `test.csv` and build `DataLoader`s.
  2. Define a small CNN (LeNet-style or with BatchNorm).
  3. Train with `CrossEntropyLoss` + `Adam` (add LR scheduler + early stopping if you want stability).
  4. Export `submission.csv` to `/kaggle/working` for direct submission.
- **Tip**: For a quick leaderboard bump, try **mild data augmentation** and optionally **Test-Time Augmentation (TTA)** on the test set.


## Data Preparation — Digit Recognizer

We load the competition CSV files and prepare them for PyTorch training:

1. **Load CSVs** → `train.csv` contains images + labels, `test.csv` only images.  
2. **Split features & labels** → `X`, `y` for training; `X_test` for final predictions.  
3. **Reshape & normalize** → images are reshaped to `(1,28,28)` and scaled to `[0,1]`.  
4. **Convert to Torch datasets** → `TensorDataset` for train/valid/test.  
5. **Train/validation split** → 80/20 random split to track generalization.  
6. **Wrap in DataLoaders** → efficient mini-batch iteration, with shuffling for training.  

This pipeline converts the CSVs into PyTorch-friendly data objects, ready for training a CNN.  


In [ ]:
# ============================
# Data Preparation
# ============================

# 1. Load CSV files
train_df = pd.read_csv("/kaggle/input/digit-recognizer/train.csv")
test_df  = pd.read_csv("/kaggle/input/digit-recognizer/test.csv")

# 2. Separate features and labels
X = train_df.drop("label", axis=1).values
y = train_df["label"].values
X_test = test_df.values

# 3. Normalize pixel values [0,1] and reshape to (N,1,28,28)
X = X.reshape(-1, 1, 28, 28).astype(np.float32) / 255.0
X_test = X_test.reshape(-1, 1, 28, 28).astype(np.float32) / 255.0
y = y.astype(np.int64)

# 4. Torch Datasets (train+valid with labels, test without labels)
from torch.utils.data import TensorDataset, random_split

full_dataset = TensorDataset(torch.tensor(X), torch.tensor(y))
test_dataset = TensorDataset(torch.tensor(X_test))

# 5. Train/validation split (80/20)
train_size = int(0.8 * len(full_dataset))
valid_size = len(full_dataset) - train_size
train_ds, valid_ds = random_split(full_dataset, [train_size, valid_size])

# 6. DataLoaders (mini-batches for training)
BATCH_SIZE = 256
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)}, Valid batches: {len(valid_loader)}, Test batches: {len(test_loader)}")


## Data Augmentation & Normalization

To improve generalization, we apply **data augmentation** during training:

- `RandomRotation(5)` → slight rotations (±5°) simulate handwriting variance.  
- `RandomAffine(..., translate=0.05)` → small translations (up to 5%) mimic shifts in digit placement.  
- `ToTensor()` → converts images to PyTorch tensors.  
- `Normalize((0.5,), (0.5,))` → scales values to have mean ≈ 0 and variance ≈ 1.  

For the test set, we only apply **tensor conversion + normalization**, without augmentation.  
This ensures stable evaluation while training sees more diverse samples.  


In [ ]:
# ============================
# Data Augmentation & Normalization
# ============================

import torchvision.transforms as transforms

# Transformations applied to training data
transform_train = transforms.Compose([
    transforms.RandomRotation(5),                     # slight rotations ±5° to simulate handwriting variations
    transforms.RandomAffine(0, translate=(0.05, 0.05)), # small shifts (up to 5%) to simulate digit misplacement
    transforms.ToTensor(),                            # convert image to tensor (C,H,W)
    transforms.Normalize((0.5,), (0.5,))              # normalize to mean=0.5, std=0.5 (roughly [-1,1] range)
])

# Transformations applied to test/validation data (no augmentation, only normalization)
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])


## CNN Model — Digit Classifier

We define a **Convolutional Neural Network (CNN)** to classify digits (0–9).  

Architecture:  
- **Convolutional layers**: three stacked conv blocks (32 → 64 → 128 filters) with ReLU activations.  
- **Pooling**: 2×2 MaxPooling after each conv to reduce spatial dimensions.  
- **Fully connected layers**:  
  - `fc1` → dense layer with 256 neurons, followed by ReLU and Dropout (p=0.3).  
  - `fc2` → final classifier with 10 outputs (one per digit).  
- **Dropout**: helps reduce overfitting by randomly disabling neurons during training.  

This is a lightweight CNN that balances performance and efficiency, well-suited for MNIST-sized images.  


In [ ]:
# ============================
# CNN Architecture for Digit Recognition
# ============================

import torch.nn as nn
import torch.nn.functional as F

class DigitCNN(nn.Module):
    def __init__(self):
        super(DigitCNN, self).__init__()
        
        # --- Convolutional layers ---
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)    # input: 1x28x28 → output: 32x28x28
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)   # 32x14x14 → 64x14x14 (after pooling)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)  # 64x7x7 → 128x7x7 (after pooling)
        
        # --- Pooling layer ---
        self.pool = nn.MaxPool2d(2, 2)  # halves H and W at each step
        
        # --- Fully connected layers ---
        self.fc1 = nn.Linear(128 * 3 * 3, 256)  # flatten output (after conv+pool) → dense layer
        self.dropout = nn.Dropout(0.3)          # dropout to reduce overfitting
        self.fc2 = nn.Linear(256, 10)           # final layer, 10 classes (digits 0–9)

    def forward(self, x):
        # --- Forward pass ---
        x = self.pool(F.relu(self.conv1(x)))  # conv1 → relu → pool
        x = self.pool(F.relu(self.conv2(x)))  # conv2 → relu → pool
        x = self.pool(F.relu(self.conv3(x)))  # conv3 → relu → pool
        x = x.view(x.size(0), -1)             # flatten (batch_size, features)
        x = F.relu(self.fc1(x))               # fully connected + relu
        x = self.dropout(x)                   # apply dropout during training
        x = self.fc2(x)                       # output logits (10 classes)
        return x


## Loss Function, Optimizer & Scheduler

We set up the training components:  

- **Device selection** → automatically uses GPU (`cuda`) if available, otherwise falls back to CPU.  
- **Loss function** → `CrossEntropyLoss` (standard for multi-class classification).  
- **Optimizer** → `Adam` with learning rate 0.001 (good default for CNNs).  
- **Scheduler** → `ReduceLROnPlateau`, which halves the learning rate when validation loss stops improving (patience = 1 epoch).  

This setup balances fast convergence with adaptivity, avoiding overfitting and poor generalization.  


In [ ]:
# ============================
# Device & Model Setup
# ============================

# Select device (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize model on the selected device
model = DigitCNN().to(device)

# ============================
# Loss Function & Optimizer
# ============================

# Cross-entropy loss for multi-class classification
criterion = nn.CrossEntropyLoss()

# Adam optimizer (good default for CNNs)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Learning rate scheduler: reduce LR if validation loss plateaus
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=1
)


## Training Loop with Early Stopping & Scheduler

This block implements the **training loop** for our CNN:

- **Epochs** → we train for up to 30 epochs.  
- **Training phase** → forward pass, loss calculation, backpropagation, optimizer step.  
- **Validation phase** → evaluate model performance on unseen validation data.  
- **Scheduler** → reduces the learning rate (`ReduceLROnPlateau`) if validation loss does not improve.  
- **Early stopping** → stops training if validation loss does not improve for 5 consecutive trials.  
- **Checkpointing** → the best model (lowest validation loss) is saved as `best_model.pt`.  

This ensures efficient training, prevents overfitting, and keeps the best-performing model.  


In [ ]:
# ============================
# Training Loop with Early Stopping & Scheduler
# ============================

EPOCHS = 30
best_val_loss = float("inf")  # track the best validation loss
patience, trials = 5, 0       # early stopping settings

for epoch in range(EPOCHS):
    # --- Training phase ---
    model.train()
    train_loss, correct, total = 0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        
        optimizer.zero_grad()       # reset gradients
        out = model(xb)             # forward pass
        loss = criterion(out, yb)   # compute loss
        loss.backward()             # backpropagation
        optimizer.step()            # update weights
        
        # accumulate training metrics
        train_loss += loss.item() * xb.size(0)
        _, preds = torch.max(out, 1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)
    train_acc = correct / total
    train_loss /= total
    
    # --- Validation phase ---
    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for xb, yb in valid_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)
            val_loss += loss.item() * xb.size(0)
            _, preds = torch.max(out, 1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    val_acc = correct / total
    val_loss /= total
    
    # --- Scheduler step (reduce LR if plateau) ---
    scheduler.step(val_loss)
    
    # --- Early stopping & checkpointing ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        trials = 0
        torch.save(model.state_dict(), "best_model.pt")  # save best model
    else:
        trials += 1
        if trials >= patience:
            print("Early stopping at epoch", epoch+1)
            break
    
    # --- Logging ---
    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.2f}% | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc*100:.2f}%")


## Load Best Model Checkpoint

After training with early stopping, the model with the **lowest validation loss** was saved as  
`best_model.pt`.  

Here we:  
- **Load the best weights** back into the model.  
- Switch the model to **evaluation mode (`eval()`)** → this disables dropout and batch norm updates, ensuring deterministic inference.  

This guarantees that any further validation, testing, or submission uses the best checkpoint found during training.  


In [ ]:
# ============================
# Load Best Model
# ============================

# Load the saved model weights (best checkpoint during training)
model.load_state_dict(torch.load("best_model.pt"))

# Set model to evaluation mode:
# - disables dropout
# - fixes batchnorm statistics
model.eval()


## Test-Time Augmentation (TTA) for Submission

To improve generalization on the unseen test set, we apply **Test-Time Augmentation (TTA)**:  

- During inference, each test image is evaluated in **multiple transformed versions**.  
- We use:  
  - `RandomRotation(5)` → small rotations (±5°).  
  - `RandomAffine(..., translate=0.05)` → slight translations.  
  - Original image (no transformation).  
- Predictions are averaged across all versions (**probability averaging**).  

This reduces the risk of misclassification on digits that are slightly rotated or shifted.  
Finally, we generate `submission.csv` in the correct format for Kaggle.  


In [ ]:
# ==============================
#  Test-Time Augmentation (TTA)
# ==============================

tta_transforms = [
    transforms.Compose([transforms.RandomRotation(5), transforms.ToTensor()]),       # rotated images
    transforms.Compose([transforms.RandomAffine(0, translate=(0.05, 0.05)), transforms.ToTensor()]) # shifted images
]

model.eval()
preds = []

with torch.no_grad():
    for xb in test_loader:
        xb = xb[0]  # batch of images, shape (B,1,28,28)
        outputs = []
        
        # Convert each tensor image in the batch back to PIL for augmentation
        imgs_list = [transforms.ToPILImage()(img.squeeze(0)) for img in xb.cpu()]
        
        # Apply each augmentation transform
        for t in tta_transforms:
            imgs_aug = torch.stack([t(img) for img in imgs_list])  # transformed batch
            out = model(imgs_aug.to(device))
            outputs.append(out.softmax(dim=1))  # store probability distributions
        
        # Add also the original, untransformed images
        out_orig = model(xb.to(device))
        outputs.append(out_orig.softmax(dim=1))
        
        # Average predictions across original + augmented versions
        mean_out = torch.stack(outputs).mean(0)
        _, pred = torch.max(mean_out, 1)
        preds.extend(pred.cpu().numpy())

# ==============================
# Generate submission file
# ==============================
submission = pd.DataFrame({
    "ImageId": np.arange(1, len(preds)+1),
    "Label": preds
})

submission.to_csv("submission.csv", index=False)
print(" Submission with TTA saved: submission.csv")



## Final Evaluation on Validation Set

Before generating the Kaggle submission, we evaluate the **best model** (reloaded from checkpoint) on the validation set:  

- Compute **validation loss** (average cross-entropy).  
- Compute **validation accuracy** (percentage of correctly classified digits).  
- Report results to check generalization before applying on the unseen test set.  

This step ensures that our saved best model (`best_model.pt`) is consistent and truly generalizes well.  


In [ ]:
# ==============================
# Final Evaluation on Validation Set
# ==============================

val_loss, correct, total = 0, 0, 0

with torch.no_grad():  # no gradient tracking during evaluation
    for xb, yb in valid_loader:
        xb, yb = xb.to(device), yb.to(device)
        
        # Forward pass
        out = model(xb)
        
        # Accumulate loss
        loss = criterion(out, yb)
        val_loss += loss.item() * xb.size(0)
        
        # Predictions
        _, preds = torch.max(out, 1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)

# Compute final averaged metrics
final_val_acc = 100. * correct / total
final_val_loss = val_loss / total

# Report results
print(f"\n Best model evaluation | Val Loss: {final_val_loss:.4f}, Val Acc: {final_val_acc:.2f}%")
